In [ ]:
#from LocalEvaluation import CustomMistral_7B
from GeminiAgent import GoogleVertexAI
from EvaluationTools import Evaluator

# Initialize the model
vertexai_gemini = GoogleVertexAI()
evaluator = Evaluator()

In [ ]:
# Run a simple test
from deepeval import assert_test
from deepeval import evaluate
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.metrics import GEval, AnswerRelevancyMetric

In [ ]:

# Read from file and create tests
tests = {
    'regular_tests': [],
    'intent': [],
}

# Include responses here
with open('reddit_responses.txt', 'r', encoding="utf-8") as chatbot_response, \
     open('reddit_responses2.txt', 'r', encoding="utf-8") as chatbot_response2, \
     open('reddit_inputs.txt', 'r', encoding="utf-8") as evaluation:
    test_input = ""
    test_intent = ""
    test_output = ""
    test_context = ""
    test_expected_output = ""
    
    for line in chatbot_response:
        if (line.startswith("Rag Context: ")):
            while (not line.startswith("---------------------")):
                line = chatbot_response.readline()
                test_context += " " + line
            line = evaluation.readline()
            while (not line.startswith("Input: ")):
                line = evaluation.readline()
            test_input = line.split(":")[1]
            test_expected_output = evaluation.readline().split(":")[1]
            
            intent_and_output = chatbot_response2.readline().split(":")
            test_intent = intent_and_output[0]
            test_output = intent_and_output[1]
            
            # Create a new test case
            test_case = LLMTestCase(
                input=test_intent,
                actual_output=test_output,
                retrieval_context=[test_context],
                expected_output=test_expected_output,
            )
            test_intent = LLMTestCase(
                input=test_input,
                actual_output=test_intent,
            )
            
            tests['regular_tests'].append(test_case)
            tests['intent'].append(test_intent)
            
            test_input = ""
            test_intent = ""
            test_output = ""
            test_context = ""
            test_expected_output = ""
            
            
        
        

In [ ]:
print(tests)

# First Tests

In [ ]:
len(tests)

first_batch = tests['regular_tests'][:5]
# Questions about supplements
supplement_results = evaluate(test_cases=first_batch, metrics=[evaluator.correctness_metric, evaluator.answer_relevancy, evaluator.answer_faithfulness, evaluator.contextual_relevancy, evaluator.contextual_precision, evaluator.contextual_recall])


# Follow-up questions about supplements (Also shows the LLM staying on task)
second_batch = tests['regular_tests'][5:17]
follow_up_results = evaluate(test_cases=second_batch, metrics=[evaluator.correctness_metric, evaluator.answer_relevancy, evaluator.answer_faithfulness, evaluator.contextual_relevancy, evaluator.contextual_precision, evaluator.contextual_recall])

# Question about supplements for certain conditions
third_batch = tests['regular_tests'][17:]
conditions_results = evaluate(test_cases=third_batch, metrics=[evaluator.correctness_metric, evaluator.answer_relevancy, evaluator.answer_faithfulness, evaluator.contextual_relevancy, evaluator.contextual_precision, evaluator.contextual_recall])



# Intent Tests

In [ ]:
# Focuses on intent

# Questions about supplements
first_batch = tests['intent'][:5]
supplement_results_intent = evaluate(test_cases=first_batch, metrics=[evaluator.intent_metric])


# Follow-up questions about supplements (Also shows the LLM staying on task)
second_batch = tests['intent'][5:17]
follow_up_results_intent = evaluate(test_cases=second_batch, metrics=[evaluator.intent_metric])

# Question about supplements for certain conditions
third_batch = tests['intent'][17:]
conditions_results_intent = evaluate(test_cases=third_batch, metrics=[evaluator.intent_metric])




# Reddit Tests

In [ ]:
reddit_batch = tests['regular_tests']
reddit_batch_intent = tests['intent']
# Reddit Questions
supplement_results = evaluate(test_cases=reddit_batch, metrics=[evaluator.correctness_metric, evaluator.answer_relevancy, evaluator.answer_faithfulness, evaluator.contextual_relevancy, evaluator.contextual_precision, evaluator.contextual_recall])
supplement_results_intent = evaluate(test_cases=reddit_batch_intent, metrics=[evaluator.intent_metric])

# Getting Scores

In [ ]:
def getScores(test_results):
    test_size = len(test_results)
    supplements_scores = [0,0,0,0,0,0,0]
    for data in test_results:
        i = 0
        metrics_data = data.metrics_data
        for metric in metrics_data:
            supplements_scores[i] += metric.score / test_size
            i += 1
    
    return supplements_scores

supplement_results_score = getScores(supplement_results.test_results)
supplement_results_score